In [ ]:
from argparse import ArgumentParser, Namespace
import Spectral_Preprocessing as sp
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
parser = ArgumentParser()
parser.add_argument("--range-ir-1", dest= "wn_range_ir_1", default = (948,1860))
parser.add_argument("--range-ir-2", dest= "wn_range_ir_2", default = (2698,3002))

parser.add_argument("--range-min-ir", dest= "range_min_ir", type = int, default = 400)
parser.add_argument("--range-max-ir", dest= "range_max_ir", type = int, default = 4000)
parser.add_argument("--interpolate-interval-ir", dest = "interpolate_interval_ir", type = int, default = 1) 

parser.add_argument("--finger-print-exl", dest= "finger_print_exl", type = bool, default = False)
parser.add_argument("--background-rule-ir", dest= "background_rule", type = str, default = 'mean')
parser.add_argument("--threshold-mean-intensity-ir", dest= "ths_mean_int_ir", type = float, default = 0)
parser.add_argument("--threshold-total-variation-intensity-ir", dest= "ths_tot_var_int_ir", type = float, default = 0)

parser.add_argument("--range-min-raman", dest= "range_min_raman", type = float, default = 400)
parser.add_argument("--range-max-raman", dest= "range_max_raman", type = float, default = 4000)
parser.add_argument("--interpolate-interval-raman", dest = "interpolate_interval_raman", type = float, default = 1) 

parser.add_argument("--rubberband", dest="rubberband", action="store_true", default=False,)
parser.add_argument("--minmax", dest="minmax", action="store_true", default=False,)
parser.add_argument("--no-interpolate", dest = "no_interpolate", action="store_true", default = False)
parser.add_argument("--nitrogen-calibration", dest="nitrogen_calibration", action="store_true", default=True,)
parser.add_argument("--si-peak", dest= "si_peak", type = float, default = 520)

parser.add_argument("--smooth", dest = "smooth", type = int, default = 0)
parser.add_argument("--baseline", dest = "baseline", type = int, default = 5)


In [ ]:
###Sample Code #1: Specify IR and Raman interpolation intervals along with min & max values
args: Namespace = parser.parse_args([]) ###Do not alter this section.
###PLEASE TYPE IN YOUR ARGUMENTS HERE
args.wn_range_ir_1 = (980, 1800)
args.wn_range_ir_2 = (2720, 2980)
args.range_min_ir = 674.9918 ###enter an integer
args.range_max_ir = 3999.809 ###enter an integer [max value is not included]
args.interpolate_interval_ir = 1.929 ###enter an integer
args.finger_print_exl = False
args.background_rule = 'total_var'
args.ths_mean_int_ir = 0.043
args.ths_tot_var_int_ir = 0.60#0.4543

args.range_min_raman = 355.555542 ###enter an integer or float 
args.range_max_raman = 3500#3895.935059 ##enter an integer or float [max value is not included] 
args.interpolate_interval_raman = 2.168021680216802 ###enter an integer or float

args.rubberband = 'specy' ###True for rubberband, False for iModPoly
args.minmax = 'snv' ###False for SVT normalization, True for minmax
args.no_interpolate = False ###use sample code 2 if using True for this option.
args.nitrogen_calibration = True ##True for calibration, False for no calibration.
args.si_peak = 520 ###enter an integer or float

#directory_name = '/Users/eochoa/Documents/Research_Ambuj/M4AM/spectral_libraries/MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
directory_name = 'MEC_02_115/Raw Data/' ###directory where the PTIR files will be downloaded from.
ir_export_filename =  "no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes
file_list_tmp = [] ###option to list out the files you want included. if left as-is, all files in directory will be included.

In [ ]:
np.round([0.2444870346428872,
 0.3538630179837895,
 0.43341034152723096,
 0.5054791541247465,
 0.5201756827640186,
 0.5490600480915059,
 0.5724102720120241],2)

In [ ]:
np.round([0.22817324629238317,
 0.44572111330273206,
 0.5478553977438755,
 0.60956677847845,
 0.6270516178987495,
 0.6366224290497706],2)

In [ ]:
path = 'MEC_02_115/Raw Data/'
dirs = os.listdir(path)

for d in dirs:
    if d[-4:] == 'ptir' or d==".DS_Store":
        del dirs[np.where(np.array(dirs) == d)[0][0]]

In [ ]:
files_list = []
for d in dirs:
    files = os.listdir(path + d)
    files_list = files_list + [d+'/'+f for f in files if f[-5:] == '.ptir']

In [ ]:
#files_list[1:]
files_list

In [ ]:
raman_final, ir_final, label_list, XX_ir, XX_raman, ids, coords, _, rng, ir_bg_label = sp.ptir_to_interpolated_processed_csv(args, directory_name,
                                                            ir_export_filename, raman_export_filename, 
                                                            file_list = files_list[1:], transpose = True)


In [ ]:
raman_final.shape, ir_final.shape

In [ ]:
#ids_file_name = np.array([x[19:-5] 
#                 if x[7] == '/' else x[20:-5] 
#                 for x in ids])
ids_file_name = np.array([x[19:-13] + x[-9:-5] 
                 if x[7] == '/' else x[20:-17] + '_' + '-'+x[-17:-13] + x[-9:-5] # x[20:-5] 
                 for x in ids])
ids_file_name = ids_file_name.astype('<U20')
ids_file_name_list = np.unique(ids_file_name)

idx_ = []
i = 1
for j, x in enumerate(ids_file_name):
    #idx_.append(i)
    ids_file_name[j] = x +'_'+ str(i)
    if j < ids_file_name.shape[0]-1 and x != ids_file_name[j+1]:
        i = 1
    else:
        i = i + 1

ids_file_name

In [ ]:
preposs_data_ir = pd.DataFrame(ir_final)
preposs_data_ir.columns = XX_ir
preposs_data_ir = pd.concat([preposs_data_ir, coords], axis=1)
preposs_data_ir['id'] = ids_file_name
preposs_data_ir['background'] = ir_bg_label
preposs_data_ir
preposs_data_ir.to_csv('gloves_contamination/data_ir_new_total_var_25.csv')

In [ ]:
preposs_data_raman = pd.DataFrame(raman_final)
preposs_data_raman.columns = XX_raman
preposs_data_raman = pd.concat([preposs_data_raman, coords], axis=1)
preposs_data_raman['id'] = ids_file_name
preposs_data_raman
#preposs_data_raman.to_csv('gloves_contamination/data_raman_new_no_bc.csv')

In [ ]:
np.mean(ir_bg_label == 'background')

In [ ]:
for i in range(preposs_data_ir.shape[0]):
    if ir_bg_label[i] == 'background':
        continue
    _ = plt.plot(preposs_data_ir.iloc[i,:-4])    
    _ = plt.show()

In [ ]:
for i in range(preposs_data_ir.shape[0]):
    if ir_bg_label[i] != 'background':
        continue
    _ = plt.plot(preposs_data_ir.iloc[i,:-4])    
    _ = plt.show()

In [ ]:
sum(ir_bg_label == 'background')

In [ ]:
X_ir = np.array(preposs_data_ir.iloc[:,:-3])
np.abs(np.diff(X_ir,1)).mean(1)

wn = np.array(preposs_data_ir.columns[:-3])

idx = np.logical_or(np.logical_and(wn > 980, wn < 1800),
np.logical_and(wn > 2720, wn < 2970) #2993.64990234375-10
) 

X_ir_2 = X_ir[:,idx]
#X_ir_2 = (X_ir_2 - X_ir_2.mean(1)[:,None]) / X_ir_2.std(1)[:,None]
#idx = np.logical_or(np.logical_and(wn > wn.min()+10, wn < 1860-10),
#np.logical_and(wn > 2700+10, wn < wn.max()-10) #2993.64990234375-10
#) 

#idx = [True] * wn.shape[0]

In [ ]:
idx_adpt = np.logical_or(np.logical_and(wn > 1300, wn < 1800),
np.logical_and(wn > 2720, wn < 2970) #2993.64990234375-10
) 

950 - 1850

In [ ]:
_ = plt.hist(X_ir_2.mean(1), bins=50)

In [ ]:
for i in np.argsort(X_ir_2.mean(1))[:100]:
    print(X_ir_2.mean(1)[i])
    plt.plot(preposs_data_ir.columns[:-3][idx],
             X_ir_2[i,:])
    plt.show()

In [ ]:
for i in np.argsort(X_ir[:,idx_adpt].mean(1))[:100]:
    print(X_ir[:,idx_adpt].mean(1)[i])
    plt.plot(preposs_data_ir.columns[:-3][idx_adpt],
             X_ir[:,idx_adpt][i,:])
    plt.show()

In [ ]:
_ = plt.plot(preposs_data_ir.columns[:-3][idx_adpt],
             X_ir[:,idx_adpt].T)

In [ ]:
idx_signal = np.abs(np.diff(X_ir_2,1)).mean(1) < .2

In [ ]:
plt.hist(np.abs(np.diff(X_ir[:,:],1)).mean(1), alpha=.5)
plt.hist(np.abs(np.diff(X_ir_2,1)).mean(1), alpha=.5)

In [ ]:
np.abs(np.diff(X_ir_2,1)).mean(1).mean()

In [ ]:
np.abs(np.diff(X_ir,1)).mean(1).mean()

In [ ]:
_ = plt.plot(preposs_data_ir.columns[:-3][idx],
             X_ir_2[idx_signal,:].T)

In [ ]:
_ = plt.plot(preposs_data_ir.columns[:-3][idx],
             X_ir_2[~idx_signal,:].T)

In [ ]:
((np.abs(np.diff(X_ir_2,1)).mean(1) > .3).sum(),
(np.abs(np.diff(X_ir_2,1)).mean(1) < .3).sum())

In [ ]:
df_bg_list = []
for f in os.listdir('Background_gloves'):
    df = pd.read_csv('Background_gloves/'+f)
    df = df.set_index('X value [cm⁻¹]').T
    df = df.reset_index(drop=True)
    df.insert(0, 'id', [f[27:] + '_' + str(i) for i in range(df.shape[0])])
    df_bg_list.append(df)
    
df_bg = pd.concat(df_bg_list, axis=0).reset_index(drop=True)
df_bg

df_nbg_list = []
for f in os.listdir('Non-Background_gloves'):
    df = pd.read_csv('Non-Background_gloves/'+f)
    df = df.set_index('X value [cm⁻¹]').T
    df = df.reset_index(drop=True)
    df.insert(0, 'id', [f[27:] + '_' + str(i) for i in range(df.shape[0])])
    df_nbg_list.append(df)
    
df_nbg = pd.concat(df_nbg_list, axis=0).reset_index(drop=True)
df_nbg

In [ ]:
df_bg

X_ir_bg = np.array(df_bg.iloc[:,1:])

wn_bg = np.array(df_bg.columns[1:])

idx_bg = np.logical_or(np.logical_and(wn_bg > 980, wn_bg < 1170),
                       np.logical_or(np.logical_and(wn_bg > 1290, wn_bg < 1800),
                       np.logical_and(wn_bg > 2720, wn_bg < 2950))) #2993.64990234375-10
#) 
#980-1170, 1290-1800, 2720-2950

#idx_bg = np.logical_or(np.logical_and(wn_bg > 980, wn_bg < 1170),
#                       np.logical_and(wn_bg > 1290, wn_bg < 1800),
#                       np.logical_and(wn_bg > 2720, wn_bg < 2950) 
#) 

#980-1170, 1290-1800, 2720-2950

In [ ]:
_ = plt.plot(wn_bg[idx_bg],
             X_ir_bg[:, idx_bg].T)

In [ ]:
idx_mean_int = np.logical_and(wn_bg[idx_bg] > 900, wn_bg[idx_bg] < 1800)

In [ ]:
np.nanmean(X_ir_bg,1)

In [ ]:
np.nanmean(X_ir_bg[:, idx_bg][:,idx_mean_int],1) #> 0.01

In [ ]:
np.nanquantile(np.nanmean(X_ir_bg[:, idx_bg][:,idx_mean_int],1), .9)

In [ ]:
plt.hist(np.nanmean(X_ir_bg[:, idx_bg][:,idx_mean_int],1), alpha=.5)

In [ ]:


X_ir_bg_2 = X_ir_bg[:, idx_bg]#[:,idx_mean_int]

X_ir_bg_2 = (X_ir_bg_2 - X_ir_bg_2.mean(1)[:,None]) / X_ir_bg_2.std(1)[:,None]


plt.hist(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), alpha=.5)

In [ ]:
total_var = np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1)
total_var = total_var[~np.isnan(total_var)]
total_var
#pd.DataFrame(total_var).to_csv('gloves_contamination/total_var_freq_distr.csv')

In [ ]:
np.quantile(total_var, 0.01)

In [ ]:
np.nanquantile(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), .01)

In [ ]:
np.nanquantile(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), .01) 

In [ ]:
[(alpha, np.nanquantile(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), alpha)) for alpha in [0,.01,.03,.05,.07,.1,.15,.20,.25]] 


In [ ]:
[np.nanquantile(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), alpha) for alpha in [0,.01,.03,.05,.07,.1,.15,.20,.25]] 

In [ ]:
[np.nanquantile(np.nanmean(np.abs(np.diff(X_ir_bg_2,1)), 1), alpha) for alpha in [0,.01,.03,.05,.07,.1,.15,.20,.25]] 

In [ ]:
for i in np.argsort(np.abs(np.diff(X_ir_2,1)).mean(1)):
    if np.abs(np.diff(X_ir_2,1)).mean(1)[i] > .3:
        break
    print(np.abs(np.diff(X_ir_2,1)).mean(1)[i],
          np.abs(np.diff(X_ir_2,1)).std(1)[i],)
    plt.plot(X_ir_2[i,:])
    #plt.plot(np.abs(np.diff(X_ir[i,:])))
    #plt.show()
    #plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    plt.show()

In [ ]:
for i in np.argsort(np.abs(np.diff(X_ir_2,1)).mean(1))[::-1]:
    if np.abs(np.diff(X_ir_2,1)).mean(1)[i] < .3:
        break
    print(np.abs(np.diff(X_ir_2,1)).mean(1)[i],
          np.abs(np.diff(X_ir_2,1)).std(1)[i],)
    plt.plot(X_ir_2[i,:])
    #plt.plot(np.abs(np.diff(X_ir[i,:])))
    #plt.show()
    #plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    plt.show()

In [ ]:
_ = plt.plot(preposs_data_raman.iloc[:,:-3].T)

In [ ]:
plt.hist((preposs_data_raman.iloc[:,:-3] == 65535.0).sum(1), bins=50)

In [ ]:
1239/1500

In [ ]:
(pd.DataFrame((preposs_data_raman.iloc[:,:-3] == 65535.0).sum(1)).
 reset_index().
 groupby(0).
 count().
 iloc[:,:].to_csv('gloves_contamination/distribution_saturation.csv'))

In [ ]:
pd.DataFrame((preposs_data_raman.iloc[:,:-3] == 65535.0).sum(1)).reset_index().groupby(0).count().iloc[:,:]

In [ ]:
plt.plot(pd.DataFrame((preposs_data_raman.iloc[:,:-3] == 65535.0).sum(1)).reset_index().groupby(0).count().iloc[1:40,:])#[:,None]


In [ ]:
X_raman = np.array(preposs_data_raman.iloc[:,:-3])
#mode_ram = stats.mode(np.abs(np.diff(X_raman,1)), axis=1, keepdims=False)

wn_raman = np.array(preposs_data_raman.columns[:-3])

idx_raman = np.logical_and(wn_raman > 355 + 20, wn_raman < 3895 - 100)

In [ ]:
X_raman_bc = np.array(preposs_data_raman.iloc[:,:-3])
#mode_ram = stats.mode(np.abs(np.diff(X_raman,1)), axis=1, keepdims=False)

wn_raman = np.array(preposs_data_raman.columns[:-3])

idx_raman = np.logical_and(wn_raman > 355 + 20, wn_raman < 3895 - 100)

In [ ]:
y_ = X_raman.mean(0) - 3
ci = X_raman.std(0)

_ = plt.plot(y_)
_ = plt.fill_between(range(y_.shape[0]), (y_-ci), (y_+ci), color = 'b', alpha=.2)


y_ = np.nanmean(X_raman_bc, 0) + 3
ci = np.nanstd(X_raman_bc, 0)

_ = plt.plot(y_)
_ = plt.fill_between(range(y_.shape[0]), (y_-ci), (y_+ci), color = 'orange', alpha=.2)

#_ = plt.plot(np.nanmean(X_raman_bc, 0).T)

In [ ]:
_ = plt.plot(X_raman.T)

In [ ]:
_ = plt.plot(X_raman_bc.T)

In [ ]:
for i in np.argsort((np.abs(np.diff(X_raman[:,idx_raman],1)) < 0.0001).sum(1))[::-1]:
    print(i,(np.abs(np.diff(X_raman[:,idx_raman],1)) < 0.0001).sum(1)[i])
    plt.plot(wn_raman[idx_raman],
             X_raman[i,:][idx_raman])
    #plt.plot(np.abs(np.diff(X_ir[i,:])))
    plt.show()
    #plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    #plt.show()

In [ ]:
for i in np.argsort((np.abs(np.diff(X_raman_bc[:,idx_raman],1)) < 0.0001).sum(1))[::-1]:
    print(i,(np.abs(np.diff(X_raman_bc[:,idx_raman],1)) < 0.0001).sum(1)[i])
    plt.plot(wn_raman[idx_raman],
             X_raman_bc[i,:][idx_raman])
    #plt.plot(np.abs(np.diff(X_ir[i,:])))
    plt.show()
    #plt.hist(np.abs(np.diff(X_ir,1))[i,:])
    #plt.show()